In [11]:
import pandas as pd 
import numpy as np 
import joblib

#pehle dataset load
df=pd.read_csv(r'D:\desert-agritech\data\processed\desert_hydroponic_processed.csv')

print("Dataset Loaded")
df.head()

Dataset Loaded


,temperature,humidity,soil_moisture,pH_level,light_hours,water_given_ml,nutrient_level,co2_ppm,day_number,yield_kg,water_stress,heat_stress_index,water_efficiency,pH_deviation,growth_score
0,42.490802,21.809066,42.418449,6.499340,11.839986,136.948336,63.814457,887.514105,27.902084,4.943280,0,33.223955,3.154151,0.000660,7.555623
1,54.014286,24.203070,39.974726,6.993494,7.476096,124.301492,45.929245,431.122774,9.533600,0.413325,1,40.941171,3.033614,0.493494,3.433714
2,49.639879,35.636422,30.569235,6.625334,8.773118,341.676780,96.449852,889.808273,12.372330,6.423662,1,31.950002,10.823094,0.125334,8.461659
3,46.973170,20.200132,56.436000,5.666605,11.306245,486.446529,21.897845,471.734885,17.260402,3.918849,1,37.484528,8.469366,0.833395,2.475824
4,38.120373,36.089491,48.597450,5.871160,9.856715,301.088524,58.785642,968.275391,19.328800,5.886870,0,24.362924,6.070645,0.628840,5.794333


In [12]:
# Har crop ki ideal conditions (research-based ranges)
# Format: [temp_min, temp_max, humidity_min, humidity_max, 
#          soil_min, soil_max, pH_min, pH_max, 
#          light_min, light_max, water_min, water_max,
#          nutrient_min, nutrient_max]

crops_db = {
    "Wheat (गेहूं)":          [35, 45, 10, 30, 30, 50, 6.0, 7.5, 8, 12, 150, 300, 50, 80],
    "Barley (जौ)":            [38, 50, 10, 25, 20, 40, 6.5, 7.5, 6, 10, 100, 250, 40, 70],
    "Sorghum (ज्वार)":        [35, 48, 15, 30, 25, 45, 6.0, 7.5, 8, 12, 120, 280, 40, 70],
    "Pearl Millet (बाजरा)":   [38, 50, 10, 25, 20, 40, 6.0, 7.5, 8, 12, 100, 250, 30, 60],
    "Finger Millet (रागी)":   [35, 45, 15, 30, 30, 50, 5.5, 7.0, 8, 12, 150, 300, 40, 70],
    "Quinoa (क्विनोआ)":       [35, 45, 15, 35, 30, 55, 6.0, 8.0, 8, 12, 150, 300, 50, 80],
    "Maize (मक्का)":          [35, 45, 20, 35, 40, 60, 5.8, 7.0, 10, 13, 250, 400, 60, 90],

    "Chickpea (चना)":         [35, 45, 10, 30, 30, 50, 6.0, 7.5, 8, 11, 150, 280, 30, 60],
    "Mung Bean (मूंग)":       [35, 45, 15, 35, 35, 55, 6.2, 7.2, 8, 12, 180, 320, 30, 60],
    "Lentil (मसूर)":          [35, 45, 15, 30, 35, 55, 6.0, 7.5, 8, 11, 150, 300, 30, 60],
    "Cowpea (लोबिया)":        [38, 48, 15, 35, 30, 50, 6.0, 7.5, 8, 12, 150, 300, 30, 60],
    "Black Gram (उड़द)":      [35, 45, 15, 35, 35, 55, 6.2, 7.2, 8, 12, 180, 320, 30, 60],
    "Pigeon Pea (अरहर)":      [35, 45, 15, 35, 35, 55, 6.0, 7.5, 8, 12, 180, 320, 30, 60],
    "Fenugreek (मेथी)":       [35, 42, 20, 35, 45, 70, 6.0, 7.0, 8, 11, 250, 400, 50, 80],

    "Spinach (पालक)":         [35, 42, 25, 40, 50, 75, 6.0, 7.0, 9, 12, 280, 420, 60, 90],
    "Lettuce (सलाद पत्ता)":   [35, 42, 25, 40, 55, 78, 5.8, 6.8, 10, 12, 280, 450, 60, 90],
    "Kale (करम साग)":         [35, 42, 25, 40, 55, 78, 5.8, 6.8, 9, 12, 280, 420, 60, 90],
    "Swiss Chard (चुकंदर पत्ता)": [35, 42, 25, 40, 50, 75, 6.0, 7.0, 9, 12, 280, 420, 60, 90],
    "Arugula (रॉकेट सलाद)":   [35, 42, 25, 40, 50, 75, 6.0, 7.0, 9, 12, 250, 400, 50, 80],
    "Amaranth Leaves (चौलाई)": [38, 46, 20, 38, 45, 70, 6.0, 7.5, 9, 12, 250, 400, 50, 80],

    "Mint (पुदीना)":          [35, 42, 30, 45, 60, 80, 6.0, 7.0, 9, 12, 300, 450, 60, 90],
    "Coriander (धनिया)":      [35, 42, 25, 40, 50, 75, 6.0, 7.0, 9, 12, 250, 400, 50, 80],
    "Basil (तुलसी)":          [35, 44, 25, 40, 50, 75, 5.8, 6.8, 10, 13, 250, 400, 50, 80],
    "Parsley (अजमोद)":        [35, 42, 25, 40, 50, 75, 6.0, 7.0, 9, 12, 250, 400, 50, 80],
    "Thyme (अजवायन)":         [38, 46, 15, 30, 35, 55, 6.0, 7.5, 10, 13, 150, 280, 30, 60],

    "Tomato (टमाटर)":         [35, 45, 25, 40, 50, 70, 6.0, 6.8, 12, 14, 300, 450, 60, 90],
    "Cucumber (खीरा)":        [35, 44, 30, 45, 55, 78, 5.8, 6.8, 11, 14, 320, 470, 60, 90],
    "Bell Pepper (शिमला मिर्च)": [35, 44, 25, 40, 50, 72, 6.0, 6.8, 11, 14, 300, 450, 60, 90],
    "Chili Pepper (मिर्च)":   [35, 46, 20, 38, 45, 68, 6.0, 7.0, 11, 14, 280, 430, 60, 90],
    "Eggplant (बैंगन)":       [35, 45, 25, 40, 50, 70, 5.8, 6.8, 11, 14, 300, 450, 60, 90],
    "Bottle Gourd (लौकी)":    [35, 46, 25, 40, 50, 72, 6.0, 7.0, 11, 14, 300, 450, 60, 90],
    "Bitter Gourd (करेला)":   [35, 46, 25, 40, 50, 72, 6.0, 7.0, 11, 14, 300, 450, 60, 90],

    "Radish (मूली)":          [35, 42, 25, 40, 55, 78, 5.8, 6.8, 9, 12, 280, 430, 50, 80],
    "Carrot (गाजर)":          [35, 42, 25, 40, 55, 78, 5.8, 6.8, 9, 12, 280, 430, 50, 80],
    "Onion (प्याज)":          [35, 44, 20, 35, 45, 68, 6.0, 7.0, 10, 13, 250, 400, 50, 80],
    "Garlic (लहसुन)":         [35, 44, 20, 35, 45, 68, 6.0, 7.0, 10, 13, 250, 400, 50, 80],
    "Turnip (शलजम)":          [35, 42, 25, 40, 50, 75, 5.8, 6.8, 9, 12, 280, 430, 50, 80],
    "Sweet Potato (शकरकंद)":  [38, 48, 20, 35, 40, 65, 5.5, 6.8, 10, 13, 250, 400, 50, 80],

    "Date Palm seedling (खजूर)": [40, 50, 10, 25, 25, 45, 6.5, 8.0, 10, 14, 150, 300, 30, 60],
    "Aloe Vera (एलोवेरा)":    [38, 50, 10, 25, 20, 40, 6.5, 8.0, 10, 14, 100, 250, 20, 50],
    "Moringa (सहजन)":         [38, 50, 10, 30, 25, 45, 6.0, 7.5, 10, 14, 150, 300, 30, 60],
    "Cactus Pear (नागफनी)":   [40, 50, 5, 20, 15, 35, 6.0, 8.0, 10, 14, 80, 200, 10, 40],
    "Jojoba (जोजोबा)":        [40, 50, 10, 25, 20, 40, 6.5, 8.0, 10, 14, 100, 250, 20, 50],
    "Henna (मेहंदी)":         [38, 48, 15, 30, 25, 45, 6.0, 7.5, 10, 14, 150, 280, 30, 60],
    "Guar (ग्वार)":           [38, 48, 10, 25, 20, 40, 6.5, 7.5, 9, 13, 120, 250, 30, 60],
    "Sesame (तिल)":           [35, 46, 15, 30, 25, 45, 6.0, 7.5, 9, 13, 130, 270, 30, 60],
    "Castor (अरंडी)":         [35, 48, 15, 30, 25, 45, 6.0, 7.5, 9, 13, 130, 270, 30, 60],

    "Alfalfa (रिजका)":        [35, 45, 20, 35, 40, 65, 6.2, 7.5, 9, 13, 250, 400, 50, 80],
    "Cotton (कपास)":          [35, 46, 15, 30, 35, 58, 5.8, 7.0, 10, 13, 200, 350, 40, 70],
    "Saffron (केसर)":         [10, 25, 30, 50, 50, 75, 6.0, 8.0, 8, 11, 200, 350, 40, 70],
}

print(f"Total crops in database: {len(crops_db)}")

Total crops in database: 50


In [13]:
# Parameter weights (total = 1.0)
WEIGHTS = {
    'temp':     0.25,  # Desert mai sabse critical
    'humidity': 0.10,
    'soil':     0.15,
    'ph':       0.20,  # Plant nutrition ke liye critical
    'light':    0.15,
    'water':    0.10,
    'nutrient': 0.05
}

def calculate_match_score(value, min_val, max_val):
    if min_val <= value <= max_val:
        center = (min_val + max_val) / 2
        range_half = (max_val - min_val) / 2
        if range_half == 0:
            return 100.0
        centrality = 1 - (abs(value - center) / range_half)
        return round(85 + (centrality * 15), 1)
    elif value < min_val:
        diff = min_val - value
        range_size = max_val - min_val
        if range_size == 0:
            return 0.0
        return round(max(0, 85 - (diff / range_size) * 85), 1)
    else:
        diff = value - max_val
        range_size = max_val - min_val
        if range_size == 0:
            return 0.0
        return round(max(0, 85 - (diff / range_size) * 85), 1)


def recommend_crops(temperature, humidity, soil_moisture, pH_level,
                    light_hours, water_given_ml, nutrient_level, top_n=5):
    results = []

    for crop, ranges in crops_db.items():
        temp_min, temp_max, hum_min, hum_max, soil_min, soil_max, \
        ph_min, ph_max, light_min, light_max, water_min, water_max, \
        nut_min, nut_max = ranges

        # Weighted score
        weighted_score = (
            calculate_match_score(temperature, temp_min, temp_max)    * WEIGHTS['temp'] +
            calculate_match_score(humidity, hum_min, hum_max)         * WEIGHTS['humidity'] +
            calculate_match_score(soil_moisture, soil_min, soil_max)  * WEIGHTS['soil'] +
            calculate_match_score(pH_level, ph_min, ph_max)           * WEIGHTS['ph'] +
            calculate_match_score(light_hours, light_min, light_max)  * WEIGHTS['light'] +
            calculate_match_score(water_given_ml, water_min, water_max) * WEIGHTS['water'] +
            calculate_match_score(nutrient_level, nut_min, nut_max)   * WEIGHTS['nutrient']
        )

        results.append({
            'crop': crop,
            'match_score': round(weighted_score, 1)
        })

    # Sort karo
    results = sorted(results, key=lambda x: x['match_score'], reverse=True)

    # Normalize — top crop = 100%
    top_score = results[0]['match_score']
    for r in results:
        r['match_score'] = round((r['match_score'] / top_score) * 100, 1)

    return results[:top_n]

print("Weighted scoring ready!")

Weighted scoring ready!


In [14]:
# Desert conditions test
temperature = 42.0
humidity = 25.0
soil_moisture = 60.0
pH_level = 6.5
light_hours = 12.0
water_given_ml = 300.0
nutrient_level = 80.0

print("=== INPUT CONDITIONS ===")
print(f"Temperature  : {temperature}°C")
print(f"Humidity     : {humidity}%")
print(f"Soil Moisture: {soil_moisture}%")
print(f"pH Level     : {pH_level}")
print(f"Light Hours  : {light_hours} hrs")
print(f"Water Given  : {water_given_ml} ml")
print(f"Nutrient     : {nutrient_level}%")

print("\n=== TOP 10 RECOMMENDED CROPS ===")
recommendations = recommend_crops(
    temperature, humidity, soil_moisture,
    pH_level, light_hours, water_given_ml,
    nutrient_level, top_n=10
)

for i, rec in enumerate(recommendations):
    print(f"{i+1}. {rec['crop']} → {rec['match_score']}% match")

=== INPUT CONDITIONS ===
Temperature  : 42.0°C
Humidity     : 25.0%
Soil Moisture: 60.0%
pH Level     : 6.5
Light Hours  : 12.0 hrs
Water Given  : 300.0 ml
Nutrient     : 80.0%

=== TOP 10 RECOMMENDED CROPS ===
1. Chili Pepper (मिर्च) → 100.0% match
2. Bottle Gourd (लौकी) → 99.3% match
3. Bitter Gourd (करेला) → 99.3% match
4. Onion (प्याज) → 99.2% match
5. Garlic (लहसुन) → 99.2% match
6. Amaranth Leaves (चौलाई) → 98.8% match
7. Maize (मक्का) → 98.2% match
8. Sweet Potato (शकरकंद) → 98.2% match
9. Eggplant (बैंगन) → 97.7% match
10. Bell Pepper (शिमला मिर्च) → 97.4% match


In [15]:

# crops_db ko file mein save karo
import pickle
with open(r'D:\desert-agritech\models\crops_db.pkl', 'wb') as f:
    pickle.dump(crops_db, f)

print("Crops database saved!")
print("Ready to integrate with Flask API!")

Crops database saved!
Ready to integrate with Flask API!


In [16]:
# Har crop ke liye extra details add karo
crops_extra = {
    "Wheat (गेहूं)":          {"min_days": 90,  "max_days": 120, "sowing_depth": "3-5 cm", "water_schedule": "Every 10 days", "harvest_tip": "Harvest when golden yellow"},
    "Barley (जौ)":            {"min_days": 85,  "max_days": 110, "sowing_depth": "3-5 cm", "water_schedule": "Every 12 days", "harvest_tip": "Harvest when heads droop"},
    "Sorghum (ज्वार)":        {"min_days": 90,  "max_days": 120, "sowing_depth": "2-4 cm", "water_schedule": "Every 10 days", "harvest_tip": "Harvest when seeds are hard"},
    "Pearl Millet (बाजरा)":   {"min_days": 75,  "max_days": 100, "sowing_depth": "2-3 cm", "water_schedule": "Every 8 days",  "harvest_tip": "Harvest when panicle turns brown"},
    "Finger Millet (रागी)":   {"min_days": 85,  "max_days": 110, "sowing_depth": "1-2 cm", "water_schedule": "Every 8 days",  "harvest_tip": "Harvest when grains are firm"},
    "Quinoa (क्विनोआ)":       {"min_days": 90,  "max_days": 120, "sowing_depth": "1-2 cm", "water_schedule": "Every 7 days",  "harvest_tip": "Harvest when seeds rattle"},
    "Maize (मक्का)":          {"min_days": 80,  "max_days": 110, "sowing_depth": "4-5 cm", "water_schedule": "Every 7 days",  "harvest_tip": "Harvest when silk turns brown"},
    "Chickpea (चना)":         {"min_days": 90,  "max_days": 120, "sowing_depth": "5-8 cm", "water_schedule": "Every 12 days", "harvest_tip": "Harvest when pods turn yellow"},
    "Mung Bean (मूंग)":       {"min_days": 60,  "max_days": 90,  "sowing_depth": "3-5 cm", "water_schedule": "Every 6 days",  "harvest_tip": "Harvest when pods turn black"},
    "Lentil (मसूर)":          {"min_days": 80,  "max_days": 110, "sowing_depth": "3-5 cm", "water_schedule": "Every 10 days", "harvest_tip": "Harvest when lower pods turn yellow"},
    "Cowpea (लबिया)":        {"min_days": 60,  "max_days": 90,  "sowing_depth": "3-5 cm", "water_schedule": "Every 7 days",  "harvest_tip": "Harvest when pods are firm"},
    "Black Gram (उड़द)":      {"min_days": 70,  "max_days": 90,  "sowing_depth": "3-5 cm", "water_schedule": "Every 7 days",  "harvest_tip": "Harvest when pods turn black"},
    "Pigeon Pea (अरहर)":      {"min_days": 120, "max_days": 180, "sowing_depth": "5-7 cm", "water_schedule": "Every 12 days", "harvest_tip": "Harvest when 80% pods are brown"},
    "Fenugreek (मेथी)":       {"min_days": 25,  "max_days": 40,  "sowing_depth": "1-2 cm", "water_schedule": "Every 5 days",  "harvest_tip": "Harvest young leaves at 25 days"},
    "Spinach (पालक)":         {"min_days": 30,  "max_days": 50,  "sowing_depth": "1-2 cm", "water_schedule": "Every 4 days",  "harvest_tip": "Harvest outer leaves first"},
    "Lettuce (सलाद पत्ता)":   {"min_days": 30,  "max_days": 60,  "sowing_depth": "0.5 cm", "water_schedule": "Every 3 days",  "harvest_tip": "Harvest before bolting"},
    "Kale (करम साग)":         {"min_days": 55,  "max_days": 75,  "sowing_depth": "1-2 cm", "water_schedule": "Every 4 days",  "harvest_tip": "Harvest outer leaves"},
    "Swiss Chard (चुकंदर पत्ता)": {"min_days": 50, "max_days": 70, "sowing_depth": "2 cm", "water_schedule": "Every 4 days",  "harvest_tip": "Cut leaves 5cm from base"},
    "Arugula (रॉकेट सलाद)":   {"min_days": 25,  "max_days": 40,  "sowing_depth": "0.5 cm", "water_schedule": "Every 3 days",  "harvest_tip": "Harvest before flowering"},
    "Amaranth Leaves (चौलाई)": {"min_days": 30, "max_days": 50,  "sowing_depth": "1 cm",   "water_schedule": "Every 4 days",  "harvest_tip": "Harvest young tender leaves"},
    "Mint (पुदीना)":          {"min_days": 20,  "max_days": 35,  "sowing_depth": "0.5 cm", "water_schedule": "Every 3 days",  "harvest_tip": "Harvest before flowering"},
    "Coriander (धनिया)":      {"min_days": 25,  "max_days": 45,  "sowing_depth": "1-2 cm", "water_schedule": "Every 4 days",  "harvest_tip": "Harvest when 50% plants flower"},
    "Basil (तुलसी)":          {"min_days": 25,  "max_days": 40,  "sowing_depth": "0.5 cm", "water_schedule": "Every 3 days",  "harvest_tip": "Pinch off flower buds"},"Parsley (अजमोद)":        {"min_days": 70,  "max_days": 90,  "sowing_depth": "0.5 cm", "water_schedule": "Every 4 days",  "harvest_tip": "Harvest outer stems first"},
    "Thyme (अजवायन)":         {"min_days": 60,  "max_days": 90,  "sowing_depth": "0.5 cm", "water_schedule": "Every 5 days",  "harvest_tip": "Harvest before flowering"},
    "Tomato (टमाटर)":         {"min_days": 70,  "max_days": 100, "sowing_depth": "0.5 cm", "water_schedule": "Every 4 days",  "harvest_tip": "Harvest when fully red"},
    "Cucumber (खीरा)":        {"min_days": 50,  "max_days": 70,  "sowing_depth": "1-2 cm", "water_schedule": "Every 3 days",  "harvest_tip": "Harvest when dark green"},
    "Bell Pepper (शिमला मिर्च)": {"min_days": 70, "max_days": 100, "sowing_depth": "0.5 cm", "water_schedule": "Every 4 days", "harvest_tip": "Harvest when fully colored"},
    "Chili Pepper (मिर्च)":   {"min_days": 70,  "max_days": 100, "sowing_depth": "0.5 cm", "water_schedule": "Every 4 days",  "harvest_tip": "Harvest when bright red"},
    "Eggplant (बैंगन)":       {"min_days": 70,  "max_days": 100, "sowing_depth": "0.5 cm", "water_schedule": "Every 4 days",  "harvest_tip": "Harvest when shiny and firm"},
    "Bottle Gourd (लकी)":    {"min_days": 55,  "max_days": 75,  "sowing_depth": "2-3 cm", "water_schedule": "Every 4 days",  "harvest_tip": "Harvest when tender"},
    "Bitter Gourd (करेला)":   {"min_days": 55,  "max_days": 75,  "sowing_depth": "2-3 cm", "water_schedule": "Every 4 days",  "harvest_tip": "Harvest when light green"},
    "Radish (मूली)":          {"min_days": 25,  "max_days": 40,  "sowing_depth": "1-2 cm", "water_schedule": "Every 3 days",  "harvest_tip": "Harvest before woody"},
    "Carrot (गाजर)":          {"min_days": 70,  "max_days": 90,  "sowing_depth": "0.5 cm", "water_schedule": "Every 5 days",  "harvest_tip": "Harvest when shoulder visible"},
    "Onion (प्याज)":          {"min_days": 100, "max_days": 150, "sowing_depth": "1-2 cm", "water_schedule": "Every 7 days",  "harvest_tip": "Harvest when tops fall over"},
    "Garlic (लहसुन)":         {"min_days": 90,  "max_days": 120, "sowing_depth": "5-8 cm", "water_schedule": "Every 7 days",  "harvest_tip": "Harvest when leaves turn yellow"},
    "Turnip (शलजम)":          {"min_days": 45,  "max_days": 60,  "sowing_depth": "1-2 cm", "water_schedule": "Every 4 days",  "harvest_tip": "Harvest when golf ball size"},
    "Sweet Potato (शकरकंद)":  {"min_days": 90,  "max_days": 120, "sowing_depth": "5-8 cm", "water_schedule": "Every 7 days",  "harvest_tip": "Harvest when leaves yellow"},
    "Date Palm seedling (खजूर)": {"min_days": 180, "max_days": 365, "sowing_depth": "5 cm", "water_schedule": "Every 14 days", "harvest_tip": "Harvest when dates turn brown"},
    "Aloe Vera (एलोवेरा)":    {"min_days": 180, "max_days": 365, "sowing_depth": "5 cm",   "water_schedule": "Every 14 days", "harvest_tip": "Harvest outer leaves only"},
    "Moringa (सहजन)":         {"min_days": 90,  "max_days": 180, "sowing_depth": "2-3 cm", "water_schedule": "Every 10 days", "harvest_tip": "Harvest pods when 30-45cm long"},
    "Cactus Pear (नागफनी)":   {"min_days": 120, "max_days": 180, "sowing_depth": "3-5 cm", "water_schedule": "Every 20 days", "harvest_tip": "Harvest when pads are firm"},
    "Jojoba (जोजोबा)":        {"min_days": 180, "max_days": 365, "sowing_depth": "3-5 cm", "water_schedule": "Every 15 days", "harvest_tip": "Harvest when seeds turn brown"},
    "Henna (मेहंदी)":         {"min_days": 90,  "max_days": 150, "sowing_depth": "0.5 cm", "water_schedule": "Every 10 days", "harvest_tip": "Harvest young leaves"},
    "Guar (ग्वार)":           {"min_days": 75,  "max_days": 100, "sowing_depth": "3-5 cm", "water_schedule": "Every 8 days",  "harvest_tip": "Harvest tender pods"},
    "Sesame (तिल)":           {"min_days": 80,  "max_days": 110, "sowing_depth": "1-2 cm", "water_schedule": "Every 8 days",  "harvest_tip": "Harvest before pods shatter"},"Castor (अरंडी)":         {"min_days": 90,  "max_days": 150, "sowing_depth": "3-5 cm", "water_schedule": "Every 10 days", "harvest_tip": "Harvest when capsules turn brown"},
    "Alfalfa (रिजका)":        {"min_days": 60,  "max_days": 90,  "sowing_depth": "1-2 cm", "water_schedule": "Every 5 days",  "harvest_tip": "Harvest at 10% bloom stage"},
    "Cotton (कपास)":          {"min_days": 150, "max_days": 180, "sowing_depth": "3-5 cm", "water_schedule": "Every 10 days", "harvest_tip": "Harvest when bolls open fully"},
    "Saffron (केसर)":         {"min_days": 180, "max_days": 240, "sowing_depth": "10-15 cm","water_schedule": "Every 14 days", "harvest_tip": "Harvest stigmas at dawn"},
}

print(f"Extra data added for {len(crops_extra)} crops!")

Extra data added for 50 crops!


In [17]:
# crops_db aur crops_extra merge karo
crops_combined = {}

for crop_name, ranges in crops_db.items():
    crops_combined[crop_name] = {
        'ranges': ranges,
        'extra': crops_extra.get(crop_name, {})
    }

print(f"Combined database: {len(crops_combined)} crops")
print("\nSample entry:")
print("Chili Pepper:", crops_combined["Chili Pepper (मिर्च)"])

# Save karo
import pickle
with open(r'D:\desert-agritech\models\crops_combined.pkl', 'wb') as f:
    pickle.dump(crops_combined, f)

print("\nCombined database saved!")

Combined database: 50 crops

Sample entry:
Chili Pepper: {'ranges': [35, 46, 20, 38, 45, 68, 6.0, 7.0, 11, 14, 280, 430, 60, 90], 'extra': {'min_days': 70, 'max_days': 100, 'sowing_depth': '0.5 cm', 'water_schedule': 'Every 4 days', 'harvest_tip': 'Harvest when bright red'}}

Combined database saved!


In [18]:
# Check karo kaunse crops ka extra data missing hai
missing = []
for crop in crops_combined:
    if not crops_combined[crop]['extra']:
        missing.append(crop)

print("Missing extra data:")
for m in missing:
    print(f"  - '{m}'")

Missing extra data:
  - 'Cowpea (लोबिया)'
  - 'Bottle Gourd (लौकी)'


In [19]:
# Missing 2 crops ka data add karo
crops_combined['Cowpea (लोबिया)']['extra'] = {
    'min_days': 60, 'max_days': 90,
    'sowing_depth': '3-5 cm',
    'water_schedule': 'Every 7 days',
    'harvest_tip': 'Harvest when pods are firm'
}

crops_combined['Bottle Gourd (लौकी)']['extra'] = {
    'min_days': 55, 'max_days': 75,
    'sowing_depth': '2-3 cm',
    'water_schedule': 'Every 4 days',
    'harvest_tip': 'Harvest when tender'
}

# Dobara save karo
with open(r'D:\desert-agritech\models\crops_combined.pkl', 'wb') as f:
    pickle.dump(crops_combined, f)

print("Fixed and saved!")

Fixed and saved!
